##Exercícios

1) Compare o desempenho desta rede com o apresentado originalmente na part #1 (notebook anterior).

2) Pode-se realizar mudanças nos hiperparâmetros com o intuito de melhorar os resultados obtidos. Ao observar os gráficos gerados ao final do treinamento, pode-se observar que durante as 20 épocas, o modelo não alcançou a convergência, que seria representada por uma linha quase horizontal no final do gráfico. Pode-se aumentrar o número de épocas para 30, 50, 100 ou maior para se tentar melhor acurária. Deve-se tomar cuidado com o overfitting, característico de um treinamento excessivo.

4) Utilizando os conhecimentos e as experiências adiquiridos nos exercícios com redes neurais anteriores, proceda a sintonia da rede, objetivando uma melhor eficiência. Mantendo a arquitetura da rede, processa a sintonia dos hiperparâmetros: `learning_rate` e `batch_size `tentando uma melhor desempenho da rede. Tome cuidado com o *overfitting*! Elabore gráficos e/ou tabelas comparativas.

5) Crie 30 figuras de dígitos (0 à 9) feitas por 3 pessoas distintas e carrege no banco de dados de testes. Faça as predições e veirifique se a rede consegue predizer positivamente seus dígitos. Qual a acurácia na predição das 30 amostras?

6) Implemente uma função para mostrar um único dígito do banco de dados de teste e suas previsões. Por exemplo: mostrar 36 amostras do dígito '9' com suas previsões.

In [ ]:
# Importando as bibliotecas
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Bibliotecas do TensorFlow/Keras
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD

# Bibliotecas para avaliação
from sklearn.metrics import confusion_matrix, classification_report

# Para medir o tempo de treinamento
import time

In [ ]:
# Carregando o banco de dados MNIST
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)
print('X_test:', X_test.shape)
print('y_test:', y_test.shape)

In [ ]:
# Verificando as classes existentes
print("Classes:", np.unique(y_train))

In [ ]:
# Função para visualizar algumas imagens do dataset
def show_data(examples, targets):
    samples = 64

    # Escolhe uma posição aleatória
    inicio = np.random.randint(0, examples.shape[0] - samples)

    plt.figure(figsize=(8, 8))

    for i in range(samples):
        plt.subplot(8, 8, i + 1)
        plt.imshow(examples[inicio + i], cmap='gray')
        plt.xlabel(f'Label: {targets[inicio + i]}')
        plt.xticks([])
        plt.yticks([])

    plt.tight_layout()
    plt.show()

show_data(X_train, y_train)

In [ ]:
# Visualizando a quantidade de exemplos de cada classe
plt.figure(figsize=(8, 5))
sns.countplot(x=y_train)

plt.xlabel('Dígito')
plt.ylabel('Quantidade')
plt.title('Distribuição das classes do MNIST')
plt.show()

In [ ]:
# Transformando as imagens de 28x28 para 784 posições
X_train_flat = X_train.reshape(-1, 28 * 28)
X_test_flat = X_test.reshape(-1, 28 * 28)

print("X_train_flat:", X_train_flat.shape)
print("X_test_flat:", X_test_flat.shape)

In [ ]:
# Normalizando os valores dos pixels
X_train_sc = X_train_flat / 255.0
X_test_sc = X_test_flat / 255.0

print("Valor mínimo:", X_train_sc.min())
print("Valor máximo:", X_train_sc.max())

In [ ]:
from tensorflow.keras.utils import to_categorical

# Convertendo os rótulos para One-Hot Encoding
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)

print("Rótulo original:", y_train[0])
print("Rótulo convertido:", y_train_cat[0])

In [ ]:
# Criando o modelo
model = Sequential()

# Camada de entrada
model.add(Input(shape=(784,)))

# Camadas ocultas
model.add(Dense(512, activation='relu'))
model.add(Dense(256, activation='relu'))
model.add(Dense(128, activation='relu'))
model.add(Dense(32, activation='relu'))

# Camada de saída
model.add(Dense(10, activation='softmax'))

In [ ]:
# Definindo o otimizador
optimizer = SGD(learning_rate=0.01)

# Compilando o modelo
model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

In [ ]:
model.summary()

In [ ]:
# Medindo o tempo de treinamento
inicio = time.time()

# Treinamento
history = model.fit(
    X_train_sc,
    y_train_cat,
    batch_size=128,
    epochs=25,
    validation_split=0.1,
    verbose=1
)

# Tempo de treinamento
tempo = time.time() - inicio

print(f"\nTempo de treinamento: {tempo:.2f} segundos")
print(f"Tempo de treinamento: {tempo/60:.2f} minutos")

In [ ]:
# Gráfico da função de custo
plt.figure(figsize=(8, 5))

plt.plot(history.history['loss'], label='Treinamento')
plt.plot(history.history['val_loss'], label='Validação')

plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.title('Função de custo durante o treinamento')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Gráfico da função de custo
plt.figure(figsize=(8, 5))

plt.plot(history.history['loss'], label='Treinamento')
plt.plot(history.history['val_loss'], label='Validação')

plt.xlabel('Épocas')
plt.ylabel('Loss')
plt.title('Função de custo durante o treinamento')
plt.legend()
plt.grid()
plt.show()

In [ ]:
# Avaliando o modelo
loss, accuracy = model.evaluate(X_test_sc, y_test_cat, verbose=0)

print(f"Acurácia no conjunto de teste: {accuracy * 100:.2f}%")
print(f"Loss no conjunto de teste: {loss:.4f}")

In [ ]:
# Realizando as predições
prediction = model.predict(X_test_sc)

# Obtendo a classe com maior probabilidade
y_predict = np.argmax(prediction, axis=1)

print("Primeiras previsões:")
print(y_predict[:20])

In [ ]:
# Escolhendo uma amostra
amostra = 9

print("Probabilidades:", prediction[amostra])
print("Soma das probabilidades:", prediction[amostra].sum())
print("Classe prevista:", prediction[amostra].argmax())
print("Probabilidade:", prediction[amostra].max() * 100)
print("Classe verdadeira:", y_test[amostra])

In [ ]:
# Criando a matriz de confusão
confusion = confusion_matrix(y_test, y_predict)

print(confusion)

In [ ]:
# Visualizando a matriz de confusão
plt.figure(figsize=(9, 7))

plt.imshow(confusion, cmap='Blues')

plt.title('Matriz de Confusão')
plt.xlabel('Classe prevista')
plt.ylabel('Classe verdadeira')

plt.colorbar()

plt.xticks(range(10))
plt.yticks(range(10))

# Adicionando os valores dentro da matriz
for i in range(10):
    for j in range(10):
        plt.text(j, i, confusion[i, j],
                 ha='center',
                 va='center')

plt.show()

In [ ]:
print(classification_report(y_test, y_predict))

In [ ]:
# Função para visualizar as imagens e suas previsões
def show_predict(examples, targets, predictions):
    plt.figure(figsize=(10, 10))

    for i in range(36):
        plt.subplot(6, 6, i + 1)

        plt.imshow(examples[i], cmap='gray')

        plt.xlabel(
            f'Real: {targets[i]} | Previsto: {predictions[i]}'
        )

        plt.xticks([])
        plt.yticks([])

    plt.tight_layout()
    plt.show()

# Visualizando as primeiras 36 imagens
show_predict(X_test, y_test, y_predict)

##Exercício 1.

In [ ]:
print(f"Acurácia da rede Part #2: {accuracy * 100:.2f}%")

In [ ]:
accuracy_part1 = 0.00  # coloque aqui a acurácia da Part #1

print(f"Part #1: {accuracy_part1 * 100:.2f}%")
print(f"Part #2: {accuracy * 100:.2f}%")

##Exercício 2.

In [ ]:
# Criando novamente a rede
model_50 = Sequential([
    Input(shape=(784,)),
    Dense(512, activation='relu'),
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(32, activation='relu'),
    Dense(10, activation='softmax')
])

# Compilando
model_50.compile(
    loss='categorical_crossentropy',
    optimizer=SGD(learning_rate=0.01),
    metrics=['accuracy']
)

# Treinando por 50 épocas
history_50 = model_50.fit(
    X_train_sc,
    y_train_cat,
    batch_size=128,
    epochs=50,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Avaliando o novo modelo
loss_50, accuracy_50 = model_50.evaluate(
    X_test_sc,
    y_test_cat,
    verbose=0
)

print(f"Acurácia com 50 épocas: {accuracy_50 * 100:.2f}%")

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(history.history['val_accuracy'], label='25 épocas')
plt.plot(history_50.history['val_accuracy'], label='50 épocas')

plt.xlabel('Épocas')
plt.ylabel('Acurácia de validação')
plt.title('Comparação entre 25 e 50 épocas')

plt.legend()
plt.grid()
plt.show()

##Exercício 4.

In [ ]:
# Função para criar a mesma arquitetura
def criar_modelo(learning_rate):
    model = Sequential([
        Input(shape=(784,)),
        Dense(512, activation='relu'),
        Dense(256, activation='relu'),
        Dense(128, activation='relu'),
        Dense(32, activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(
        loss='categorical_crossentropy',
        optimizer=SGD(learning_rate=learning_rate),
        metrics=['accuracy']
    )

    return model

In [ ]:
resultados = []

configuracoes = [
    (0.1, 128),
    (0.01, 128),
    (0.001, 128),
    (0.01, 64),
    (0.01, 256)
]

for learning_rate, batch_size in configuracoes:

    print(f"\nTestando: learning_rate={learning_rate}, batch_size={batch_size}")

    modelo = criar_modelo(learning_rate)

    history_temp = modelo.fit(
        X_train_sc,
        y_train_cat,
        batch_size=batch_size,
        epochs=10,
        validation_split=0.1,
        verbose=0
    )

    loss_temp, acc_temp = modelo.evaluate(
        X_test_sc,
        y_test_cat,
        verbose=0
    )

    resultados.append([
        learning_rate,
        batch_size,
        acc_temp * 100
    ])

In [35]:
resultados = []

configuracoes = [
    (0.1, 128),
    (0.01, 128),
    (0.001, 128),
    (0.01, 64),
    (0.01, 256)
]

for learning_rate, batch_size in configuracoes:

    print(f"\nTestando: learning_rate={learning_rate}, batch_size={batch_size}")

    modelo = criar_modelo(learning_rate)

    history_temp = modelo.fit(
        X_train_sc,
        y_train_cat,
        batch_size=batch_size,
        epochs=10,
        validation_split=0.1,
        verbose=0
    )

    loss_temp, acc_temp = modelo.evaluate(
        X_test_sc,
        y_test_cat,
        verbose=0
    )

    resultados.append([
        learning_rate,
        batch_size,
        acc_temp * 100
    ])


Testando: learning_rate=0.1, batch_size=128

Testando: learning_rate=0.01, batch_size=128

Testando: learning_rate=0.001, batch_size=128

Testando: learning_rate=0.01, batch_size=64

Testando: learning_rate=0.01, batch_size=256


##Exercício 5.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

ValueError: mount failed

In [ ]:
from PIL import Image
from numpy import asarray

# Caminho das imagens
path = '/content/drive/MyDrive/Colab Notebooks/images/'

In [ ]:
# Carregando um exemplo
img = Image.open(path + '0.png')

# Convertendo para 28x28
img = img.resize((28, 28))

# Convertendo para escala de cinza
img = img.convert('L')

# Convertendo para array
img = asarray(img)

# Invertendo as cores para ficar semelhante ao MNIST
img = 255 - img

plt.imshow(img, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
# Vetores para armazenar as imagens
novas_imagens = []
novos_labels = []

for i in range(10):

    img = Image.open(path + f'{i}.png')

    # Redimensionando
    img = img.resize((28, 28))

    # Escala de cinza
    img = img.convert('L')

    # Convertendo para array
    img = asarray(img)

    # Invertendo as cores
    img = 255 - img

    novas_imagens.append(img)
    novos_labels.append(i)

# Convertendo para numpy
novas_imagens = np.array(novas_imagens)
novos_labels = np.array(novos_labels)

print(novas_imagens.shape)

In [ ]:
# Transformando 28x28 em 784
novas_imagens_flat = novas_imagens.reshape(-1, 784)

# Normalizando
novas_imagens_sc = novas_imagens_flat / 255.0

In [ ]:
# Realizando as predições
prediction_new = model.predict(novas_imagens_sc)

# Obtendo a classe prevista
y_predict_new = np.argmax(prediction_new, axis=1)

print("Real:    ", novos_labels)
print("Previsto:", y_predict_new)

In [ ]:
# Calculando a acurácia
acertos = np.sum(novos_labels == y_predict_new)

accuracy_new = acertos / len(novos_labels)

print(f"Acertos: {acertos}/{len(novos_labels)}")
print(f"Acurácia: {accuracy_new * 100:.2f}%")

##Exercício 6.

In [ ]:
def mostrar_digito(digito, quantidade=36):

    # Seleciona as imagens do dígito escolhido
    indices = np.where(y_test == digito)[0][:quantidade]

    plt.figure(figsize=(10, 10))

    for i, indice in enumerate(indices):

        plt.subplot(6, 6, i + 1)

        plt.imshow(X_test[indice], cmap='gray')

        plt.xlabel(
            f'Real: {y_test[indice]} | '
            f'Previsto: {y_predict[indice]}'
        )

        plt.xticks([])
        plt.yticks([])

    plt.tight_layout()
    plt.show()

In [ ]:
mostrar_digito(9)

In [ ]:
mostrar_digito(5)

In [ ]:
mostrar_digito(2)

Conclusão

Após o treinamento da rede neural Feedforward, foi possível realizar a classificação das imagens do banco de dados MNIST utilizando uma arquitetura totalmente conectada. O processamento das imagens foi realizado por meio da transformação das matrizes de 28 × 28 pixels em vetores de 784 posições, seguida da normalização dos valores dos pixels.

A rede foi composta por quatro camadas ocultas, utilizando a função de ativação ReLU, e uma camada de saída com 10 neurônios e função Softmax. O treinamento foi realizado utilizando o algoritmo SGD e a função de custo categorical_crossentropy.

A avaliação utilizando o conjunto de teste permitiu verificar a capacidade de generalização do modelo por meio da acurácia, matriz de confusão e métricas de classificação. Também foram realizadas alterações nos hiperparâmetros, permitindo observar como diferentes valores de learning rate, batch size e número de épocas influenciam o desempenho da rede.

Por fim, foram realizadas predições utilizando imagens de dígitos desenhados manualmente, permitindo verificar o comportamento da rede diante de imagens diferentes das presentes originalmente no banco MNIST.